# Capa Silver

### Aqui vamos a generar todas las limpiezas y transformaciones de nuestros dataframes guardardos "raw" o crudos en capa_bronze, una vez finalizado vamos a guardarlas en formato delta limpias y transformadas , tambien se opto por generar un enriquecimiento mediante "joins"

## 1 - Importacion de modulos
Se importan los modulos especificos

In [0]:
# SILVER - Procesamiento de datos
from pyspark.sql import *
from pyspark.sql.functions import *

## 2 - Cargo los Dataframes de capa bronze cada uno a su respectiva variable que los almacena


In [0]:
accounts_df = spark.sql(f"""select * from capa_bronze.tables.account""")
customers_df = spark.sql(f"""select * from capa_bronze.tables.customers""")
loans_df= spark.sql(f"""select * from capa_bronze.tables.loans""")
loans_payments_df  = spark.sql(f"""select * from capa_bronze.tables.loan_payments""")
transactions_df= spark.sql(f"""select * from capa_bronze.tables.transactions""")

> ### * Corroboramos que la tabla _customers_ sigue sucia y sin transformar los datos

In [0]:
spark.sql(
    """
        select * from capa_bronze.tables.customers
    """
).display()

## 3 - Limpieza y transformación de _customers_

1: Se limpian los tildes de cada letra vocal, se crea una funcion de limpieza y luego con _withColumn_ llamamos a la funcion especifica _remove_accents_

2: Se crea una funcion que valida con una expresion regular (Regex) el email y si no se respeta, se rellena con null

3:  Se Extraer _'street'_, _'street_number'_, _'city'_ de _address_ ya que puede causar errores tipicos al estar calle, numero y provincia en la misma columna, se desglosan en 3 para mayor precisión.
Luego borramos la columna _address_ porque no se requiere mas

4: Calculamos la edad al dia de la fecha de cada cliente y lo agregamos al dataframe

5: Eliminamos duplicados y convertimos tipos (solo date), se convierte solo _date_ ya que al guardar los datos previamente (en Pipeline) en Azure SQL, ya tenemos las columnas con sus respectivos tipos de datos

#### 6: Preservar clientes activos y datos críticos aunque falte información en algunos campos secundarios (teléfono o email), el registro puede representar un cliente activo con operaciones importantes, entonces optamos en este caso, mantener todo cliente por mas que tengan datos faltantes y/o adulterados, luego esto se tiene que informar a su sector especifico para tomar la accion necesaria.

Crear columnas en base a los datos nulos, solo 2 en este ejemplo de trabajo: _phone_number_ e _email_ ,para mayor informacion se podria crear las restantes

In [0]:
# EXPLICAR LIMPIEZA DE TILDES
def remove_accents(col_name):
    return regexp_replace(
        regexp_replace(
            regexp_replace(
                regexp_replace(
                    regexp_replace(
                        col(col_name), "á", "a"
                    ), "é", "e"
                ), "í", "i"
            ), "ó", "o"
        ), "ú", "u"
    )

# Validar y limpiar correos electrónicos
def validate_email(email_col):
    return when(col(email_col).rlike(r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'), col(email_col)).otherwise(lit(None))

customers_df = customers_df.withColumn("name", remove_accents("name")) \
       .withColumn("address", remove_accents("address")) \
       .withColumn("email", lower(col("email"))) \
       .withColumn("email", validate_email("email"))

# Extraemos 'street', 'street_number', 'city' de address
customers_df = customers_df.withColumn("street", regexp_extract(col("address"), r"^(.*?)\s\d+", 1)) \
       .withColumn("street_number", regexp_extract(col("address"), r"(\d+)", 1)) \
       .withColumn("city", substring_index(col("address"), "-", -1).alias("city")) \
       .withColumn("city", trim(col("city")))

customers_df = customers_df.drop("address")

# 4. Calculamos edad
customers_df = customers_df.withColumn("birth_date", col("birth_date").cast("date")) \
       .withColumn("age", (datediff(current_date(), col("birth_date")) / 365).cast("int"))

# Limpieza de datos de ventas: eliminamos duplicados y convertimos tipos
customers_df = (customers_df.dropDuplicates()
                            .withColumn("birth_date", col("birth_date").cast("date"))
                  )

# Crear columnas en base a los datos nulos 
#No se eliminan filas por ser clientes activos, pero sí se etiqueta la calidad del dato.

customers_df = customers_df \
    .withColumn("has_phone", when(col("phone_number").isNotNull(), lit(True)).otherwise(lit(False))) \
    .withColumn("has_email", when(col("email").isNotNull(), lit(True)).otherwise(lit(False)))

customers_df.display()

> ### * Corroboramos que la tabla _accounts_ sigue sucia y sin transformar los datos

In [0]:
# accounts_df
spark.sql(
    """
        select * from capa_bronze.tables.account
    """
    ).display()

## 4 - Limpieza y transformación de _accounts_

1: Se eliminan duplicados para mantener vigente reglas ACID: Integridad y Consistencia.

2: Al consultar la tabla vemos que hay un error en la palabra dolares (d[?]lares) entonces ,antes de hacer el reemplazo en todas las palabras que contengan _tilde_ corregimos el error principal en  d[?]lares

#### 3: Aqui estamos en la misma disyuntiva de antes, creamos columnas extras para cada valor nulo que poseamos. 
Si la cuenta tiene un balance válido y un _created_date_, probablemente es una cuenta real activa. Eliminarla puede perder información útil.
Procedemos a quedarnos con los nulos pero informar a estos con un agregado de columna

4: Estandarizamos el texto de la columna completa _class_of_account_ ya que provenian caracteres problematicos desde la ingesta nativa

In [0]:
accounts_df = (accounts_df.dropDuplicates()
                            .withColumn("opened_date", col("opened_date").cast("date"))) \
                            .withColumn("created_date", col("created_date").cast("date"))

# Corregimos codificación de class_of_account "Cuenta en d?lares"
accounts_df = accounts_df.withColumn("class_of_account", regexp_replace(col("class_of_account"), "d[?¿]lares", "dolares"))


# Creamos columnas para los valores nulos que poseamos en esas mismas columnas

accounts_df = accounts_df \
    .withColumn("has_opened_date", col("opened_date").isNotNull()) \
    .withColumn("has_created_date", col("created_date").isNotNull()) \
    .withColumn("has_class_of_account", col("class_of_account").isNotNull()) \
    .withColumn("has_balance", col("balance").isNotNull())

# 3. Estandarizamos texto
accounts_df = accounts_df.withColumn("class_of_account", lower(trim(col("class_of_account"))))
accounts_df = accounts_df.withColumn("class_of_account", regexp_replace(col("class_of_account"), "á", "a")) \
       .withColumn("class_of_account", regexp_replace(col("class_of_account"), "é", "e")) \
       .withColumn("class_of_account", regexp_replace(col("class_of_account"), "í", "i")) \
       .withColumn("class_of_account", regexp_replace(col("class_of_account"), "ó", "o")) \
       .withColumn("class_of_account", regexp_replace(col("class_of_account"), "ú", "u"))

accounts_df.display()

> ### * Corroboramos que la tabla _loans_ sigue sucia y sin transformar los datos


In [0]:
spark.sql(
    """
        select * from capa_bronze.tables.loans
    """
).display()


## 4 - Limpieza y transformación de _loans_

1: Se eliminan duplicados para mantener vigente reglas ACID: Integridad y Consistencia.

2: Se cambia tipo de dato en _start_date_ a date

3: Convertimos columna _amount_ a decimal y la renombramos como _loan_amount_ para mejorar el entendimiento 

4: Se imputan (rellenan) los valores nulos (si hay) en _interest_rate_ con 6% pedido por el banco (tasa promedio histórica del negocio), es un valor razonable por defecto para una tasa de interés si hay valores nulos

#### 5: En start_date nulos se opto por crear una columna agregada en base a las fechas de inicio de prestamos, estas fechas son muy especificas y imputarlas con cierta fecha seria un error, agregamos una columna has_start_date_ boolean, si hay falsos que no pagaron, se notifica al sector especifico el problema

6: Se quitan espacios y pasamos a minúsculas todas las letras de _status_ para normalizar el texto

7: Una vez verificado que se clonan los valores de _amount_ a _loan_amount_ correctamente, eliminamos esa columna

In [0]:
from pyspark.sql.functions import *
# Eliminan duplicados y convertimos tipos necesarios
loans_df = (loans_df.dropDuplicates() \
    .withColumn("start_date", col("start_date").cast("date")))

# amount a decimal y renombramos como loan_amount
loans_df = loans_df.withColumn(
    "loan_amount",
    col("amount").cast("decimal(15,2)")
)

# Imputamos interest_rate con valor por defecto general bancario
loans_df = loans_df.withColumn(
    "interest_rate",
    when(col("interest_rate").isNull(), 6.0).otherwise(col("interest_rate"))
)

#Reemplazamos start_date nulo con loan_date
loans_df = loans_df \
    .withColumn("has_start_date", col("start_date").isNotNull())

# Normalizamos el texto del estado (status)
loans_df = loans_df.withColumn(
    "status",
    lower(trim(col("status")))
)

# Eliminamos columna original 'amount'
loans_df = loans_df.drop("amount")

loans_df.display()

> ### * Corroboramos que la tabla _loan_payments_ sigue sucia y sin transformar los datos


In [0]:

spark.sql(
    """
        select * from capa_bronze.tables.loan_payments
    """
).display()


## 5 - Limpieza y transformación de _loan_payments_

1: Se eliminan duplicados para mantener vigente reglas ACID: Integridad y Consistencia.

2: Convertimos columna _payment_date_ a date 

#### 3: Al haber datos nulos en una columna importante como _amount_paid_ (monto_pagado) se crea una columna especifica para conservar el dato nulo pero antes se notifica inmediatamente ya que es una condición Missing Critical Data (Datos Críticos Faltantes).
### En este caso de negocio se procede siempre mediante Data Anomaly Flagging (explicado en documentación general)

4: Se genera una nueva columna _has_payment_date_ para datos nulos en la fecha de pago

5: Se limpia la columna _payment_date_, asegurando que solo contenga valores con el formato esperado

In [0]:
from pyspark.sql.functions import col, when

loans_payments_df = (loans_payments_df.dropDuplicates() \
    .withColumn("payment_date", col("payment_date").cast("date")))

# Creamos una columna para conservar el nulo de amount paid y poner una bandera
loans_payments_df = loans_payments_df.withColumn(
    "is_amount_paid_null", when(col("amount_paid").isNull(), True).otherwise(False) 
)

# Generamos una columna nueva de has_payment_date
loans_payments_df = loans_payments_df.withColumn(
    "has_payment_date", when(col("payment_date").isNull(), True).otherwise(False) 
)

# Corrección de fechas invalidas
loans_payments_df = loans_payments_df.withColumn("payment_date", 
        when(col("payment_date").rlike("^\d{4}-\d{2}-\d{2}$"), col("payment_date"))
        .otherwise(None)) 

# AMOUNT_PAID NULOS EN DOCUMENTACION

loans_payments_df.display()

> ### * Corroboramos que la tabla _transactions_ sigue sucia y sin transformar los datos


In [0]:
spark.sql(f"""select * from capa_bronze.tables.transactions""").display()

## 6 - Limpieza y transformación de _transactions_

1: Se eliminan duplicados para mantener vigente reglas ACID: Integridad y Consistencia.

2: Convertimos columna _system_date_ y _transaction_date_ a tipo de dato _date_ 

#### 3: Despues de la ingesta, observamos fechas invalidas y/o anomalas, se puede observar que _transaction_date_ (fecha de transferencia) se genero antes que la propia "alta" de la cuenta en el sistema (_system_date_), como hicimos anteriormente, optamos por la creación de una columna bandera _date_anomaly_ y notificamos al sector correspondiente 

4: Se aplican condiciones para rellenar valores nulos en _description_ con descripciones predeterminadas según el tipo de transacción.

Si es nulo pero _type_ = Pago : Se rellena con _Pago_varios_ 

Si es nulo pero _type_ = Retiro : Se deja retiro pero se concatena _"sin detalle"_ 

In [0]:
# explicar borrar duplicados y castear/cambiar tipo de dato a date
transactions_df = transactions_df.dropDuplicates()\
    .withColumn("system_date", col("system_date").cast("date")) \
    .withColumn("transaction_date", col("transaction_date").cast("date"))

# explicar anomalias en fechas de creaciones , creacion de columna con fecha anomalas
transactions_df = transactions_df.withColumn(
    "date_anomaly",
    when(col("transaction_date") > col("system_date"), True).otherwise(False)
)

# rellenar valores nulos de la columna "description" con mensaje "Pago varios" o si es TYPE= PAGO PERO DESCRIPTION NULL = PAGO - SIN DETALLE

transactions_df = transactions_df.withColumn(
    "description",
    when(col("description").isNull() & (col("type") == "Pago"), "Pago varios")
    .when(col("description").isNull(), concat(col("type"), lit(" - sin detalle")))
    .otherwise(col("description"))
)
transactions_df.display()

## 7 - Enriquecimiento de tablas mediante _joins_


Se creo una operación de unión (join) entre las tablas de datos existentes, creando un DataFrame enriquecido (loans_enriched_df) que combina información de las múltiples fuentes relacionadas con los préstamos. 

Solo trae la información de los clientes que pidieron prestamos

* _Modelo relacional en documentación_

In [0]:
loans_enriched_df = (customers_df
                    .join(accounts_df, "customer_id")
                    .join(loans_df, "customer_id")
                    .join(transactions_df, "account_id")                    
                    .join(loans_payments_df, "loan_id")  

                     )
loans_enriched_df.printSchema(
                     )

loans_enriched_df.display()



## 8 - Creación de tablas temporales de cada DataFrame

In [0]:
accounts_df.createOrReplaceTempView("accounts")
customers_df.createOrReplaceTempView("customers")
loans_df.createOrReplaceTempView("loans")
loans_payments_df.createOrReplaceTempView("loans_payments")
transactions_df.createOrReplaceTempView("transactions")
loans_enriched_df.createOrReplaceTempView("loans_enriched")


## - Verficamos la tabla enriquecida para corroborar exactitud

In [0]:
spark.sql("""
    SELECT *
    FROM loans_enriched
""").display()


## 9 - Creamos Schemas en el catalogo especifico (capa_silver.tables) y luego listamos cada tabla dentro del mismo catalogo mediante iteracion _FOR_

In [0]:
# Creacion de esquema y tabla en silver de tabla enriquecida

spark.sql(f"""
          CREATE SCHEMA IF NOT EXISTS capa_silver.tables
          """)

# Este código crea tablas Delta Lake dentro de Databricks, en el metastore predeterminado, mediante un for iterando los nombres de las tablas
# EXPLICAR DONDE SE GUARDAN Y COMO (DELTA TABLES)


table_names = ["accounts", "customers", "loans", "loans_payments", "transactions", "loans_enriched"]

for table in table_names:
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS capa_silver.tables.{table}
        AS SELECT * FROM {table}
    """)     



Corroboramos que se haya creado una tabla X en la metastore y que tenga los datos correctos

In [0]:

spark.sql(
    """
        select * from capa_silver.tables.loans_enriched
    """
).display()

## 10 - Guardamos en formato delta todos los DataSets ya limpios y transformados

Se uso un guardado simple no iterativo (_sin FOR_) para la visualizacion nativa del metodo de guardado

In [0]:
accounts_df.write.format("delta").mode("overwrite").save(f"abfss://silver@mistorageprincipal.dfs.core.windows.net/delta_tables/accounts")
customers_df.write.format("delta").mode("overwrite").save(f"abfss://silver@mistorageprincipal.dfs.core.windows.net/delta_tables/customers")
loans_df.write.format("delta").mode("overwrite").save(f"abfss://silver@mistorageprincipal.dfs.core.windows.net/delta_tables/loans")
loans_payments_df.write.format("delta").mode("overwrite").save(f"abfss://silver@mistorageprincipal.dfs.core.windows.net/delta_tables/loans_payments")
transactions_df.write.format("delta").mode("overwrite").save(f"abfss://silver@mistorageprincipal.dfs.core.windows.net/delta_tables/transactions")
loans_enriched_df.write.format("delta").mode("overwrite").save(f"abfss://silver@mistorageprincipal.dfs.core.windows.net/delta_tables/loans_enriched")


### FINALIZACION CAPA _SILVER_
---